In [ ]:
# =============================================================================
# CELL 1: Linear Regression on All 5 LGD Datasets using run_talent_method
# =============================================================================

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset_preprocessing import preprocess_dataset_specific
from src.methods.method_runner import run_talent_method

# Store results globally for next cell
LGD_DATASETS = ['0001.heloc', '0002.loss2', '0003.axa', '0004.base_model', '0005.base_modelisation']
LGD_DATA = {}  # Will store (df, target_col, cat_cols, num_cols) for each dataset
LGD_RESULTS = {}  # Will store full results from run_talent_method

print("="*80)
print("  LINEAR REGRESSION ON ALL LGD DATASETS (via run_talent_method)")
print("="*80)

# Run on all datasets
print("\n" + "-"*80)
print(f"{'Dataset':<25} {'Samples':>8} {'Features':>10} {'R² Mean':>12} {'MSE Mean':>12} {'MAE Mean':>10}")
print("-"*80)

for dataset_name in LGD_DATASETS:
    try:
        # First load dataset to get info and store for debugging
        df, target_col, cat_cols, num_cols = preprocess_dataset_specific(
            task='lgd',
            dataset=dataset_name,
            apply_pca=True,
            remove_outliers=True
        )
        LGD_DATA[dataset_name] = (df, target_col, cat_cols, num_cols)
        
        # Run LinearRegression via run_talent_method
        fold_results = run_talent_method(
            task='lgd',
            dataset=dataset_name,
            test_size=0.2,
            val_size=0.1,
            cv_splits=5,
            seed=42,
            method='LinearRegression',
            tune=False,
            verbose=False,
        )
        
        LGD_RESULTS[dataset_name] = fold_results
        
        # Aggregate metrics across folds (use correct capitalized keys!)
        r2_scores = [fold_results[fold]['metrics']['R2'] for fold in fold_results]
        mse_scores = [fold_results[fold]['metrics']['MSE'] for fold in fold_results]
        mae_scores = [fold_results[fold]['metrics']['MAE'] for fold in fold_results]
        
        r2_mean = np.mean(r2_scores)
        r2_std = np.std(r2_scores)
        mse_mean = np.mean(mse_scores)
        mae_mean = np.mean(mae_scores)
        
        # Store aggregated stats for convenience
        LGD_RESULTS[dataset_name]['_aggregated'] = {
            'r2_mean': r2_mean,
            'r2_std': r2_std,
            'r2_scores': r2_scores,
            'mse_mean': mse_mean,
            'mse_scores': mse_scores,
            'mae_mean': mae_mean,
            'mae_scores': mae_scores,
            'n_samples': len(df),
            'n_features': len(num_cols) + len(cat_cols),
        }
        
        # Print summary
        r2_indicator = "⚠️" if r2_mean < 0.1 else "✓"
        print(f"{dataset_name:<25} {len(df):>8,} {len(num_cols)+len(cat_cols):>10} "
              f"{r2_mean:>12.4f} {mse_mean:>12.4f} {mae_mean:>10.4f} {r2_indicator}")
        
    except Exception as e:
        print(f"{dataset_name:<25} ERROR: {str(e)[:50]}")
        LGD_RESULTS[dataset_name] = {'_error': str(e)}
        import traceback
        traceback.print_exc()

print("-"*80)

# Summary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

valid_results = {k: v for k, v in LGD_RESULTS.items() if '_aggregated' in v}
datasets = list(valid_results.keys())
r2_means = [valid_results[d]['_aggregated']['r2_mean'] for d in datasets]
r2_stds = [valid_results[d]['_aggregated']['r2_std'] for d in datasets]

colors = ['#C73E1D' if r2 < 0.1 else '#F18F01' if r2 < 0.3 else '#28A745' for r2 in r2_means]
axes[0].bar(range(len(datasets)), r2_means, yerr=r2_stds, capsize=5, color=colors, edgecolor='black')
axes[0].set_xticks(range(len(datasets)))
axes[0].set_xticklabels([d.split('.')[1] for d in datasets], rotation=45, ha='right')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Linear Regression R² by Dataset')
axes[0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[0].axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='Poor threshold')
axes[0].legend()

# Box plot of R² across folds
r2_data = []
labels = []
for d in datasets:
    r2_data.extend(valid_results[d]['_aggregated']['r2_scores'])
    labels.extend([d.split('.')[1]] * len(valid_results[d]['_aggregated']['r2_scores']))

r2_df = pd.DataFrame({'Dataset': labels, 'R²': r2_data})
sns.boxplot(data=r2_df, x='Dataset', y='R²', ax=axes[1], palette='viridis')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].set_title('R² Distribution Across CV Folds')

plt.tight_layout()
plt.show()

# Print datasets needing investigation
print("\n" + "="*80)
print("  DATASETS NEEDING INVESTIGATION (R² < 0.1)")
print("="*80)
poor_datasets = [d for d in datasets if valid_results[d]['_aggregated']['r2_mean'] < 0.1]
if poor_datasets:
    for d in poor_datasets:
        print(f"  ⚠️  {d}: R² = {valid_results[d]['_aggregated']['r2_mean']:.4f}")
else:
    print("  ✓ All datasets have R² >= 0.1")

print("\n✓ Results stored in LGD_RESULTS and LGD_DATA for debugging")

In [ ]:
# =============================================================================
# CELL 2: Debug Selected Dataset(s) - DEEP ANALYSIS of Linear Regression
# =============================================================================

# -----------------------------------------------------------------------------
# SELECT DATASET(S) TO DEBUG (modify this list)
# -----------------------------------------------------------------------------
DATASETS_TO_DEBUG = ['0001.heloc']  # <-- CHANGE THIS
# -----------------------------------------------------------------------------

print("="*80)
print(f"  DEEP DEBUGGING LINEAR REGRESSION: {DATASETS_TO_DEBUG}")
print("="*80)


def debug_lr_results(dataset_name):
    """Deep debug Linear Regression results from run_talent_method."""
    
    if dataset_name not in LGD_RESULTS or '_error' in LGD_RESULTS[dataset_name]:
        print(f"\n❌ No valid results for {dataset_name}")
        return
    
    fold_results = LGD_RESULTS[dataset_name]
    agg = fold_results['_aggregated']
    df, target_col, cat_cols, num_cols = LGD_DATA[dataset_name]
    
    print(f"\n{'#'*80}")
    print(f"# DEEP DEBUGGING: {dataset_name}")
    print(f"{'#'*80}")
    
    # Aggregate predictions across all folds
    all_y_true = np.concatenate([fold_results[f]['y_true'] for f in fold_results if f != '_aggregated'])
    all_y_pred = np.concatenate([fold_results[f]['y_pred'] for f in fold_results if f != '_aggregated'])
    all_y_pred_raw = np.concatenate([fold_results[f]['y_pred_raw'] for f in fold_results if f != '_aggregated'])
    
    total_clipped_below = sum(fold_results[f]['n_clipped_below'] for f in fold_results if f != '_aggregated')
    total_clipped_above = sum(fold_results[f]['n_clipped_above'] for f in fold_results if f != '_aggregated')
    
    residuals = all_y_true - all_y_pred
    residuals_raw = all_y_true - all_y_pred_raw
    
    # =========================================================================
    # 1. OVERVIEW
    # =========================================================================
    print(f"\n[1. OVERVIEW]")
    print(f"   {'─'*60}")
    print(f"   Samples:        {agg['n_samples']:,}")
    print(f"   Features:       {agg['n_features']}")
    print(f"   R² Mean:        {agg['r2_mean']:.6f}")
    print(f"   R² Std:         {agg['r2_std']:.6f}")
    print(f"   R² per fold:    {[f'{r:.4f}' for r in agg['r2_scores']]}")
    print(f"   MSE Mean:       {agg['mse_mean']:.6f}")
    print(f"   MAE Mean:       {agg['mae_mean']:.6f}")
    
    # Check if this is a PCA-reduced dataset
    is_pca_dataset = any(col.startswith('pca_') for col in num_cols)
    print(f"\n   PCA Dataset:    {'YES ⚠️' if is_pca_dataset else 'No'}")
    if is_pca_dataset:
        n_pca_components = sum(1 for col in num_cols if col.startswith('pca_'))
        print(f"   PCA Components: {n_pca_components}")
    
    # =========================================================================
    # 2. PER-FOLD BREAKDOWN (identify problematic folds)
    # =========================================================================
    print(f"\n[2. PER-FOLD BREAKDOWN]")
    print(f"   {'─'*60}")
    print(f"   {'Fold':<6} {'R²':>10} {'MSE':>10} {'MAE':>10} {'Clip<0':>8} {'Clip>1':>8} {'Pred Range':>20}")
    print(f"   {'-'*6} {'-'*10} {'-'*10} {'-'*10} {'-'*8} {'-'*8} {'-'*20}")
    
    fold_ids = sorted([k for k in fold_results.keys() if k != '_aggregated'])
    worst_fold = None
    worst_r2 = float('inf')
    
    for fold_id in fold_ids:
        fold = fold_results[fold_id]
        r2 = fold['metrics']['R2']
        mse = fold['metrics']['MSE']
        mae = fold['metrics']['MAE']
        clip_below = fold['n_clipped_below']
        clip_above = fold['n_clipped_above']
        pred_min = fold['y_pred_raw'].min()
        pred_max = fold['y_pred_raw'].max()
        
        flag = " ⚠️" if r2 < 0 else ""
        print(f"   {fold_id:<6} {r2:>10.4f} {mse:>10.4f} {mae:>10.4f} {clip_below:>8} {clip_above:>8} [{pred_min:>7.2f}, {pred_max:>7.2f}]{flag}")
        
        if r2 < worst_r2:
            worst_r2 = r2
            worst_fold = fold_id
    
    print(f"\n   Worst fold: {worst_fold} with R² = {worst_r2:.4f}")
    
    # =========================================================================
    # 3. RAW PREDICTION ANALYSIS (before clipping)
    # =========================================================================
    print(f"\n[3. RAW PREDICTION ANALYSIS (before clipping)]")
    print(f"   {'─'*60}")
    
    print(f"   Range:          [{all_y_pred_raw.min():.4f}, {all_y_pred_raw.max():.4f}]")
    print(f"   Mean:           {all_y_pred_raw.mean():.4f}")
    print(f"   Std:            {all_y_pred_raw.std():.4f}")
    print(f"   Median:         {np.median(all_y_pred_raw):.4f}")
    
    # Percentile analysis
    percentiles = [0.1, 1, 5, 25, 50, 75, 95, 99, 99.9]
    print(f"\n   Percentiles of raw predictions:")
    for p in percentiles:
        val = np.percentile(all_y_pred_raw, p)
        flag = " ⚠️" if val < -0.5 or val > 1.5 else ""
        print(f"   {p:>6.1f}%: {val:>10.4f}{flag}")
    
    # Extreme predictions
    n_extreme_low = (all_y_pred_raw < -1).sum()
    n_extreme_high = (all_y_pred_raw > 2).sum()
    n_very_extreme_low = (all_y_pred_raw < -10).sum()
    n_very_extreme_high = (all_y_pred_raw > 10).sum()
    
    print(f"\n   Extreme predictions:")
    print(f"   - pred < -1:    {n_extreme_low:>6} ({n_extreme_low/len(all_y_pred_raw)*100:.2f}%)")
    print(f"   - pred > 2:     {n_extreme_high:>6} ({n_extreme_high/len(all_y_pred_raw)*100:.2f}%)")
    print(f"   - pred < -10:   {n_very_extreme_low:>6} ({n_very_extreme_low/len(all_y_pred_raw)*100:.2f}%)")
    print(f"   - pred > 10:    {n_very_extreme_high:>6} ({n_very_extreme_high/len(all_y_pred_raw)*100:.2f}%)")
    
    if n_very_extreme_low > 0 or n_very_extreme_high > 0:
        print(f"\n   ⚠️  EXTREME PREDICTIONS DETECTED!")
        print(f"      This indicates outliers in feature space are causing huge extrapolations.")
    
    # =========================================================================
    # 4. WORST PREDICTIONS ANALYSIS
    # =========================================================================
    print(f"\n[4. WORST PREDICTIONS ANALYSIS]")
    print(f"   {'─'*60}")
    
    # Find indices of worst predictions (largest absolute errors)
    abs_errors = np.abs(residuals_raw)
    worst_indices = np.argsort(abs_errors)[-20:][::-1]  # Top 20 worst
    
    print(f"\n   Top 20 worst predictions (raw):")
    print(f"   {'Idx':>6} {'True':>8} {'Pred_Raw':>12} {'Error':>10} {'Pred_Clip':>10}")
    print(f"   {'-'*6} {'-'*8} {'-'*12} {'-'*10} {'-'*10}")
    
    for i, idx in enumerate(worst_indices[:20]):
        true_val = all_y_true[idx]
        pred_raw = all_y_pred_raw[idx]
        pred_clip = all_y_pred[idx]
        error = residuals_raw[idx]
        print(f"   {idx:>6} {true_val:>8.4f} {pred_raw:>12.2f} {error:>+10.2f} {pred_clip:>10.4f}")
    
    # Analyze: are worst predictions associated with specific target values?
    worst_true_values = all_y_true[worst_indices[:20]]
    print(f"\n   Target values for worst predictions:")
    print(f"   - Mean:    {worst_true_values.mean():.4f}")
    print(f"   - Range:   [{worst_true_values.min():.4f}, {worst_true_values.max():.4f}]")
    print(f"   - At 0:    {(worst_true_values == 0).sum()}")
    print(f"   - At 1:    {(worst_true_values == 1).sum()}")
    
    # =========================================================================
    # 5. FEATURE SPACE ANALYSIS
    # =========================================================================
    print(f"\n[5. FEATURE SPACE ANALYSIS]")
    print(f"   {'─'*60}")
    
    feature_stats = []
    for col in num_cols[:50]:  # Limit to first 50 features
        if col in df.columns:
            series = df[col].dropna()
            if len(series) > 0:
                stats = {
                    'name': col,
                    'min': series.min(),
                    'max': series.max(),
                    'mean': series.mean(),
                    'std': series.std(),
                    'skew': series.skew() if len(series) > 2 else 0,
                    'kurtosis': series.kurtosis() if len(series) > 3 else 0,
                    'range': series.max() - series.min(),
                    'cv': series.std() / series.mean() if series.mean() != 0 else 0,
                    'pct_extreme': ((series < series.quantile(0.001)) | (series > series.quantile(0.999))).mean() * 100
                }
                feature_stats.append(stats)
    
    # Find features with extreme statistics
    print(f"\n   Features with EXTREME ranges (potential outlier issues):")
    print(f"   {'Feature':<20} {'Min':>12} {'Max':>12} {'Range':>12} {'Skew':>8} {'Kurt':>8}")
    print(f"   {'-'*20} {'-'*12} {'-'*12} {'-'*12} {'-'*8} {'-'*8}")
    
    sorted_by_range = sorted(feature_stats, key=lambda x: x['range'], reverse=True)
    for feat in sorted_by_range[:15]:
        flag = " ⚠️" if feat['range'] > 1000 or abs(feat['skew']) > 5 else ""
        print(f"   {feat['name']:<20} {feat['min']:>12.2f} {feat['max']:>12.2f} {feat['range']:>12.2f} {feat['skew']:>8.2f} {feat['kurtosis']:>8.2f}{flag}")
    
    # =========================================================================
    # 6. OUTLIER DETECTION IN FEATURE SPACE
    # =========================================================================
    print(f"\n[6. OUTLIER DETECTION IN FEATURE SPACE]")
    print(f"   {'─'*60}")
    
    # For each feature, count extreme values
    outlier_counts = {}
    for col in num_cols:
        if col in df.columns:
            series = df[col].dropna()
            if len(series) > 20:
                q1, q3 = series.quantile(0.25), series.quantile(0.75)
                iqr = q3 - q1
                if iqr > 0:
                    n_outliers = ((series < q1 - 3*iqr) | (series > q3 + 3*iqr)).sum()
                    if n_outliers > 0:
                        outlier_counts[col] = n_outliers
    
    if outlier_counts:
        print(f"\n   Features with outliers (>3 IQR from quartiles):")
        sorted_outliers = sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)
        for col, count in sorted_outliers[:15]:
            pct = count / len(df) * 100
            print(f"   - {col:<30}: {count:>5} ({pct:.2f}%)")
    else:
        print(f"   No extreme outliers detected (>3 IQR)")
    
    # Check for rows that are outliers in MANY features
    print(f"\n   Checking for rows that are outliers across multiple features...")
    row_outlier_counts = pd.Series(0, index=df.index)
    
    for col in num_cols:
        if col in df.columns:
            series = df[col]
            if series.std() > 0:
                z_scores = (series - series.mean()) / series.std()
                row_outlier_counts += (z_scores.abs() > 3).astype(int)
    
    multi_outliers = row_outlier_counts[row_outlier_counts >= 3]
    if len(multi_outliers) > 0:
        print(f"\n   Rows that are outliers in 3+ features: {len(multi_outliers)}")
        print(f"   Max features outlying in single row: {row_outlier_counts.max()}")
        
        # Show top multi-outlier rows
        top_multi = row_outlier_counts.nlargest(10)
        print(f"\n   Top 10 multi-feature outlier rows:")
        print(f"   {'Row':>8} {'# Features':>12} {'Target':>10}")
        for idx, count in top_multi.items():
            target_val = df.loc[idx, target_col] if idx in df.index else np.nan
            print(f"   {idx:>8} {count:>12} {target_val:>10.4f}")
    
    # =========================================================================
    # 7. PCA-SPECIFIC ANALYSIS (if applicable)
    # =========================================================================
    if is_pca_dataset:
        print(f"\n[7. PCA-SPECIFIC ANALYSIS]")
        print(f"   {'─'*60}")
        
        pca_cols = [col for col in num_cols if col.startswith('pca_')]
        
        print(f"\n   PCA Component Statistics:")
        print(f"   {'Component':<12} {'Min':>12} {'Max':>12} {'Mean':>10} {'Std':>10} {'Skew':>8}")
        print(f"   {'-'*12} {'-'*12} {'-'*12} {'-'*10} {'-'*10} {'-'*8}")
        
        extreme_pca_components = []
        for col in pca_cols[:20]:  # First 20 components
            if col in df.columns:
                series = df[col]
                min_val = series.min()
                max_val = series.max()
                mean_val = series.mean()
                std_val = series.std()
                skew_val = series.skew()
                
                # Flag if extreme range
                flag = ""
                if max_val - min_val > 1000:
                    flag = " ⚠️ EXTREME"
                    extreme_pca_components.append(col)
                
                print(f"   {col:<12} {min_val:>12.2f} {max_val:>12.2f} {mean_val:>10.2f} {std_val:>10.2f} {skew_val:>8.2f}{flag}")
        
        if extreme_pca_components:
            print(f"\n   ⚠️  {len(extreme_pca_components)} PCA components have EXTREME ranges!")
            print(f"      This indicates PCA outlier amplification.")
            print(f"      Rows with unusual values across many correlated features")
            print(f"      project to extreme positions in PCA space.")
            
            # Find the extreme rows in PCA space
            print(f"\n   Rows with extreme PCA values:")
            for pca_col in extreme_pca_components[:5]:
                series = df[pca_col]
                extreme_high = series.nlargest(3)
                extreme_low = series.nsmallest(3)
                print(f"\n   {pca_col}:")
                print(f"      Highest: {list(zip(extreme_high.index.tolist(), extreme_high.values.round(2)))}")
                print(f"      Lowest:  {list(zip(extreme_low.index.tolist(), extreme_low.values.round(2)))}")
    
    # =========================================================================
    # 8. TARGET ANALYSIS
    # =========================================================================
    print(f"\n[8. TARGET VARIABLE: {target_col}]")
    print(f"   {'─'*60}")
    target = df[target_col]
    
    print(f"   Min:            {target.min():.6f}")
    print(f"   Max:            {target.max():.6f}")
    print(f"   Mean:           {target.mean():.6f}")
    print(f"   Median:         {target.median():.6f}")
    print(f"   Std:            {target.std():.6f}")
    print(f"   Skewness:       {target.skew():.4f}")
    
    zero_pct = (target == 0).mean() * 100
    one_pct = (target == 1).mean() * 100
    print(f"\n   Exactly 0:      {(target == 0).sum():,} ({zero_pct:.1f}%)")
    print(f"   Exactly 1:      {(target == 1).sum():,} ({one_pct:.1f}%)")
    print(f"   In (0,1):       {((target > 0) & (target < 1)).sum():,} ({((target > 0) & (target < 1)).mean()*100:.1f}%)")
    
    # =========================================================================
    # 9. FEATURE-TARGET CORRELATIONS
    # =========================================================================
    print(f"\n[9. FEATURE-TARGET CORRELATIONS]")
    print(f"   {'─'*60}")
    
    correlations = {}
    for col in num_cols:
        if col in df.columns:
            corr = df[col].corr(target)
            if not np.isnan(corr):
                correlations[col] = corr
    
    sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    print(f"\n   Top 15 correlations with target:")
    for i, (feat, corr) in enumerate(sorted_corr[:15]):
        bar = '█' * int(abs(corr) * 30)
        print(f"   {i+1:2d}. {feat:30s} {corr:+.4f} {bar}")
    
    max_corr = max(abs(c) for _, c in sorted_corr) if sorted_corr else 0
    mean_abs_corr = np.mean([abs(c) for _, c in sorted_corr]) if sorted_corr else 0
    
    print(f"\n   Summary:")
    print(f"   - Max |correlation|:  {max_corr:.4f}")
    print(f"   - Mean |correlation|: {mean_abs_corr:.4f}")
    
    # =========================================================================
    # 10. LEVERAGE/INFLUENCE ANALYSIS
    # =========================================================================
    print(f"\n[10. LEVERAGE/INFLUENCE ANALYSIS]")
    print(f"   {'─'*60}")
    
    # Use first fold's data for this analysis
    first_fold_id = min([k for k in fold_results.keys() if k != '_aggregated'])
    fold_y_true = fold_results[first_fold_id]['y_true']
    fold_y_pred_raw = fold_results[first_fold_id]['y_pred_raw']
    fold_residuals = fold_y_true - fold_y_pred_raw
    
    # Identify high-influence points (large residual AND unusual prediction)
    pred_z = (fold_y_pred_raw - fold_y_pred_raw.mean()) / fold_y_pred_raw.std()
    resid_z = (fold_residuals - fold_residuals.mean()) / fold_residuals.std()
    
    # High influence = extreme prediction AND large residual
    influence_score = np.abs(pred_z) * np.abs(resid_z)
    
    high_influence = np.where(influence_score > 5)[0]
    print(f"   High-influence points (fold {first_fold_id}): {len(high_influence)}")
    
    if len(high_influence) > 0:
        print(f"\n   Top high-influence points:")
        top_influence_idx = np.argsort(influence_score)[-10:][::-1]
        print(f"   {'Idx':>6} {'True':>8} {'Pred':>12} {'Resid':>10} {'Influence':>10}")
        for idx in top_influence_idx:
            print(f"   {idx:>6} {fold_y_true[idx]:>8.4f} {fold_y_pred_raw[idx]:>12.2f} {fold_residuals[idx]:>+10.2f} {influence_score[idx]:>10.2f}")
    
    # =========================================================================
    # 11. VISUALIZATIONS
    # =========================================================================
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    fig.suptitle(f'Deep Debug: {dataset_name} (R²={agg["r2_mean"]:.4f})', fontsize=14, fontweight='bold')
    
    # Row 1: Predictions
    ax = axes[0, 0]
    ax.scatter(all_y_true, all_y_pred, alpha=0.3, s=10)
    ax.plot([0, 1], [0, 1], 'r--', label='Perfect')
    ax.set_xlabel('Actual LGD')
    ax.set_ylabel('Predicted LGD (clipped)')
    ax.set_title('Actual vs Predicted')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    
    ax = axes[0, 1]
    ax.scatter(all_y_true, all_y_pred_raw, alpha=0.3, s=10)
    ax.plot([0, 1], [0, 1], 'r--')
    ax.axhline(y=0, color='orange', linestyle=':', alpha=0.7)
    ax.axhline(y=1, color='orange', linestyle=':', alpha=0.7)
    ax.set_xlabel('Actual LGD')
    ax.set_ylabel('Raw Predicted')
    ax.set_title(f'Raw Predictions\n[{all_y_pred_raw.min():.1f}, {all_y_pred_raw.max():.1f}]')
    
    ax = axes[0, 2]
    ax.hist(all_y_pred_raw, bins=100, edgecolor='white', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--', alpha=0.7)
    ax.axvline(x=1, color='r', linestyle='--', alpha=0.7)
    ax.set_xlabel('Raw Prediction')
    ax.set_ylabel('Count')
    ax.set_title('Raw Prediction Distribution')
    
    ax = axes[0, 3]
    # Log scale for raw predictions to see extremes
    pred_for_log = all_y_pred_raw.copy()
    ax.hist(pred_for_log, bins=100, edgecolor='white', alpha=0.7)
    ax.set_yscale('log')
    ax.axvline(x=0, color='r', linestyle='--', alpha=0.7)
    ax.axvline(x=1, color='r', linestyle='--', alpha=0.7)
    ax.set_xlabel('Raw Prediction')
    ax.set_ylabel('Count (log scale)')
    ax.set_title('Raw Predictions (log y-axis)')
    
    # Row 2: Residuals and errors
    ax = axes[1, 0]
    ax.scatter(all_y_pred, residuals, alpha=0.3, s=10)
    ax.axhline(y=0, color='r', linestyle='--')
    ax.set_xlabel('Predicted (clipped)')
    ax.set_ylabel('Residuals')
    ax.set_title('Residuals vs Predicted')
    
    ax = axes[1, 1]
    ax.hist(residuals_raw, bins=100, edgecolor='white', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Raw Residuals')
    ax.set_ylabel('Count')
    ax.set_title(f'Raw Residual Distribution\nstd={residuals_raw.std():.2f}')
    
    ax = axes[1, 2]
    # Absolute error by true value
    ax.scatter(all_y_true, np.abs(residuals_raw), alpha=0.3, s=10)
    ax.set_xlabel('True LGD')
    ax.set_ylabel('Absolute Error')
    ax.set_title('Absolute Error by True Value')
    
    ax = axes[1, 3]
    # Per-fold R² comparison
    fold_r2s = [fold_results[f]['metrics']['R2'] for f in fold_ids]
    colors = ['red' if r < 0 else 'steelblue' for r in fold_r2s]
    ax.bar(range(len(fold_r2s)), fold_r2s, color=colors)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.axhline(y=agg['r2_mean'], color='orange', linestyle='--', label=f"Mean: {agg['r2_mean']:.3f}")
    ax.set_xlabel('Fold')
    ax.set_ylabel('R²')
    ax.set_title('R² by Fold')
    ax.legend()
    
    # Row 3: Feature analysis
    ax = axes[2, 0]
    ax.hist(target, bins=50, edgecolor='white', alpha=0.7, color='#2E86AB')
    ax.axvline(target.mean(), color='red', linestyle='--', label=f'Mean: {target.mean():.3f}')
    ax.set_xlabel('LGD')
    ax.set_ylabel('Frequency')
    ax.set_title('Target Distribution')
    ax.legend()
    
    ax = axes[2, 1]
    # Correlation bar chart
    top_corrs = sorted_corr[:10]
    feat_names = [f[0][:15] for f in top_corrs]
    corr_vals = [f[1] for f in top_corrs]
    colors = ['green' if c > 0 else 'red' for c in corr_vals]
    ax.barh(range(len(feat_names)), corr_vals, color=colors)
    ax.set_yticks(range(len(feat_names)))
    ax.set_yticklabels(feat_names)
    ax.set_xlabel('Correlation')
    ax.set_title('Top Feature Correlations')
    ax.axvline(x=0, color='black', linewidth=0.5)
    
    ax = axes[2, 2]
    # If PCA dataset, show first 2 PCA components with color = target
    if is_pca_dataset and 'pca_1' in df.columns and 'pca_2' in df.columns:
        scatter = ax.scatter(df['pca_1'], df['pca_2'], c=target, cmap='RdYlBu_r', 
                            alpha=0.5, s=10, vmin=0, vmax=1)
        plt.colorbar(scatter, ax=ax, label='LGD')
        ax.set_xlabel('PCA 1')
        ax.set_ylabel('PCA 2')
        ax.set_title('PCA Space (color=LGD)')
    else:
        # Show top 2 correlated features
        if len(sorted_corr) >= 2:
            feat1, _ = sorted_corr[0]
            feat2, _ = sorted_corr[1]
            if feat1 in df.columns and feat2 in df.columns:
                scatter = ax.scatter(df[feat1], df[feat2], c=target, cmap='RdYlBu_r',
                                    alpha=0.5, s=10, vmin=0, vmax=1)
                plt.colorbar(scatter, ax=ax, label='LGD')
                ax.set_xlabel(feat1[:20])
                ax.set_ylabel(feat2[:20])
                ax.set_title('Top 2 Features (color=LGD)')
    
    ax = axes[2, 3]
    # Influence plot
    ax.scatter(np.abs(pred_z), np.abs(resid_z), alpha=0.3, s=10)
    ax.axhline(y=2, color='r', linestyle='--', alpha=0.5)
    ax.axvline(x=2, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('|Prediction Z-score|')
    ax.set_ylabel('|Residual Z-score|')
    ax.set_title('Influence Plot (fold 0)')
    
    plt.tight_layout()
    plt.show()
    
    # =========================================================================
    # 12. DIAGNOSIS SUMMARY
    # =========================================================================
    print(f"\n[12. DIAGNOSIS SUMMARY]")
    print(f"   {'='*60}")
    
    issues = []
    recommendations = []
    r2 = agg['r2_mean']
    
    # Check R²
    if r2 < 0:
        issues.append(("CRITICAL", "NEGATIVE R²: Model worse than mean prediction"))
        recommendations.append("Linear regression fundamentally inappropriate")
    elif r2 < 0.05:
        issues.append(("SEVERE", f"VERY LOW R² ({r2:.4f}): Model explains almost nothing"))
    elif r2 < 0.15:
        issues.append(("MODERATE", f"LOW R² ({r2:.4f}): Weak linear relationship"))
    
    # Check fold variance
    r2_std = np.std(agg['r2_scores'])
    if r2_std > 0.1:
        issues.append(("MODERATE", f"HIGH FOLD VARIANCE: R² std = {r2_std:.4f}"))
        recommendations.append("Results unstable across folds - possible overfitting to outliers")
    
    # Check target distribution
    if zero_pct + one_pct > 30:
        issues.append(("MODERATE", f"BOUNDARY TARGET: {zero_pct+one_pct:.0f}% at 0 or 1"))
        recommendations.append("Consider two-part model or beta regression")
    
    # Check correlations
    if max_corr < 0.2:
        issues.append(("SEVERE", f"VERY WEAK LINEAR SIGNAL: max |r| = {max_corr:.3f}"))
        recommendations.append("Non-linear models may capture relationships better")
    elif max_corr < 0.4:
        issues.append(("MODERATE", f"WEAK LINEAR SIGNAL: max |r| = {max_corr:.3f}"))
    
    # Check extreme predictions
    extreme_pct = (n_very_extreme_low + n_very_extreme_high) / len(all_y_pred_raw) * 100
    if extreme_pct > 0.1:
        issues.append(("CRITICAL", f"EXTREME PREDICTIONS: {extreme_pct:.2f}% outside [-10, 10]"))
        recommendations.append("Outliers in feature space causing extrapolation")
    
    # Check clipping
    clipping_pct = (total_clipped_below + total_clipped_above) / len(all_y_pred) * 100
    if clipping_pct > 20:
        issues.append(("SEVERE", f"HIGH CLIPPING: {clipping_pct:.1f}% predictions outside [0,1]"))
    elif clipping_pct > 10:
        issues.append(("MODERATE", f"NOTABLE CLIPPING: {clipping_pct:.1f}% predictions outside [0,1]"))
    
    # Check PCA issues
    if is_pca_dataset and 'extreme_pca_components' in dir() and extreme_pca_components:
        issues.append(("CRITICAL", f"PCA OUTLIER AMPLIFICATION: {len(extreme_pca_components)} components have extreme range"))
        recommendations.append("Enable winsorizing in PCA or use robust PCA")
    
    # Check high-influence points
    if len(high_influence) > len(fold_y_true) * 0.05:
        issues.append(("MODERATE", f"MANY HIGH-INFLUENCE POINTS: {len(high_influence)} ({len(high_influence)/len(fold_y_true)*100:.1f}%)"))
        recommendations.append("Consider robust regression or outlier handling")
    
    # Print issues by severity
    print("\n   ISSUES FOUND:")
    for severity, msg in sorted(issues, key=lambda x: {'CRITICAL': 0, 'SEVERE': 1, 'MODERATE': 2}.get(x[0], 3)):
        icon = {'CRITICAL': '🔴', 'SEVERE': '🟠', 'MODERATE': '🟡'}.get(severity, '⚪')
        print(f"   {icon} [{severity}] {msg}")
    
    if recommendations:
        print("\n   RECOMMENDATIONS:")
        for i, rec in enumerate(recommendations, 1):
            print(f"   {i}. {rec}")
    
    print("\n   SUGGESTED NEXT STEPS:")
    print("   • Try tree-based models (XGBoost, CatBoost, RandomForest)")
    print("   • Try TabPFN (handles complex patterns well)")
    if is_pca_dataset:
        print("   • Enable PCA winsorizing in dataset_preprocessing.py")
        print("   • Consider stricter pre-PCA outlier removal")
    print("   • Review feature engineering opportunities")
    
    return {
        'r2': r2,
        'issues': issues,
        'max_corr': max_corr,
        'extreme_pct': extreme_pct,
        'clipping_pct': clipping_pct,
        'is_pca': is_pca_dataset
    }


# Run debugging
debug_results = {}
for dataset in DATASETS_TO_DEBUG:
    if dataset in LGD_DATA:
        debug_results[dataset] = debug_lr_results(dataset)
    else:
        print(f"\n❌ '{dataset}' not found. Available: {list(LGD_DATA.keys())}")

In [ ]:
# =============================================================================
# CELL 2: Debug Selected Dataset(s) - TEXT-ONLY OUTPUT for Claude debugging
# =============================================================================

# -----------------------------------------------------------------------------
# SELECT DATASET(S) TO DEBUG (modify this list)
# -----------------------------------------------------------------------------
DATASETS_TO_DEBUG = ['0001.heloc']  # <-- CHANGE THIS
# -----------------------------------------------------------------------------

def debug_lr_text_only(dataset_name):
    """Text-only deep debug for copy-paste to Claude."""
    
    output = []
    def log(msg=""):
        output.append(msg)
    
    if dataset_name not in LGD_RESULTS or '_error' in LGD_RESULTS[dataset_name]:
        log(f"❌ No valid results for {dataset_name}")
        return "\n".join(output)
    
    fold_results = LGD_RESULTS[dataset_name]
    agg = fold_results['_aggregated']
    df, target_col, cat_cols, num_cols = LGD_DATA[dataset_name]
    
    log("="*80)
    log(f"LINEAR REGRESSION DEBUG: {dataset_name}")
    log("="*80)
    
    # Aggregate predictions
    fold_ids = sorted([k for k in fold_results.keys() if k != '_aggregated'])
    all_y_true = np.concatenate([fold_results[f]['y_true'] for f in fold_ids])
    all_y_pred = np.concatenate([fold_results[f]['y_pred'] for f in fold_ids])
    all_y_pred_raw = np.concatenate([fold_results[f]['y_pred_raw'] for f in fold_ids])
    
    total_clipped_below = sum(fold_results[f]['n_clipped_below'] for f in fold_ids)
    total_clipped_above = sum(fold_results[f]['n_clipped_above'] for f in fold_ids)
    
    residuals_raw = all_y_true - all_y_pred_raw
    
    # =========================================================================
    # 1. OVERVIEW
    # =========================================================================
    log("\n[1. OVERVIEW]")
    log(f"Samples: {agg['n_samples']:,}")
    log(f"Features: {agg['n_features']}")
    log(f"R² Mean: {agg['r2_mean']:.6f}")
    log(f"R² Std: {agg['r2_std']:.6f}")
    log(f"R² per fold: {[round(r, 4) for r in agg['r2_scores']]}")
    log(f"MSE Mean: {agg['mse_mean']:.6f}")
    log(f"MAE Mean: {agg['mae_mean']:.6f}")
    
    is_pca_dataset = any(col.startswith('pca_') for col in num_cols)
    log(f"PCA Dataset: {is_pca_dataset}")
    if is_pca_dataset:
        n_pca = sum(1 for col in num_cols if col.startswith('pca_'))
        log(f"PCA Components: {n_pca}")
    
    # =========================================================================
    # 2. PER-FOLD BREAKDOWN
    # =========================================================================
    log("\n[2. PER-FOLD BREAKDOWN]")
    log(f"{'Fold':<6} {'R²':>10} {'MSE':>10} {'MAE':>10} {'Clip<0':>8} {'Clip>1':>8} {'RawPred Min':>12} {'RawPred Max':>12}")
    
    for fold_id in fold_ids:
        fold = fold_results[fold_id]
        r2 = fold['metrics']['R2']
        mse = fold['metrics']['MSE']
        mae = fold['metrics']['MAE']
        clip_below = fold['n_clipped_below']
        clip_above = fold['n_clipped_above']
        pred_min = fold['y_pred_raw'].min()
        pred_max = fold['y_pred_raw'].max()
        flag = " ⚠️" if r2 < 0 else ""
        log(f"{fold_id:<6} {r2:>10.4f} {mse:>10.4f} {mae:>10.4f} {clip_below:>8} {clip_above:>8} {pred_min:>12.2f} {pred_max:>12.2f}{flag}")
    
    # =========================================================================
    # 3. RAW PREDICTION STATISTICS
    # =========================================================================
    log("\n[3. RAW PREDICTION STATISTICS]")
    log(f"Range: [{all_y_pred_raw.min():.4f}, {all_y_pred_raw.max():.4f}]")
    log(f"Mean: {all_y_pred_raw.mean():.4f}")
    log(f"Std: {all_y_pred_raw.std():.4f}")
    log(f"Median: {np.median(all_y_pred_raw):.4f}")
    
    log("\nPercentiles:")
    for p in [0.1, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9]:
        val = np.percentile(all_y_pred_raw, p)
        flag = " ⚠️" if val < -0.5 or val > 1.5 else ""
        log(f"  {p:>5.1f}%: {val:>12.4f}{flag}")
    
    log("\nExtreme prediction counts:")
    log(f"  pred < -1:   {(all_y_pred_raw < -1).sum():>6} ({(all_y_pred_raw < -1).mean()*100:.2f}%)")
    log(f"  pred > 2:    {(all_y_pred_raw > 2).sum():>6} ({(all_y_pred_raw > 2).mean()*100:.2f}%)")
    log(f"  pred < -10:  {(all_y_pred_raw < -10).sum():>6} ({(all_y_pred_raw < -10).mean()*100:.2f}%)")
    log(f"  pred > 10:   {(all_y_pred_raw > 10).sum():>6} ({(all_y_pred_raw > 10).mean()*100:.2f}%)")
    log(f"  pred < -100: {(all_y_pred_raw < -100).sum():>6} ({(all_y_pred_raw < -100).mean()*100:.2f}%)")
    log(f"  pred > 100:  {(all_y_pred_raw > 100).sum():>6} ({(all_y_pred_raw > 100).mean()*100:.2f}%)")
    
    # =========================================================================
    # 4. WORST PREDICTIONS
    # =========================================================================
    log("\n[4. WORST 30 PREDICTIONS]")
    abs_errors = np.abs(residuals_raw)
    worst_indices = np.argsort(abs_errors)[-30:][::-1]
    
    log(f"{'Idx':>6} {'True':>8} {'RawPred':>12} {'Error':>12} {'ClipPred':>10}")
    for idx in worst_indices:
        log(f"{idx:>6} {all_y_true[idx]:>8.4f} {all_y_pred_raw[idx]:>12.2f} {residuals_raw[idx]:>+12.2f} {all_y_pred[idx]:>10.4f}")
    
    # Statistics on worst predictions
    worst_true = all_y_true[worst_indices]
    log(f"\nTarget values for worst 30:")
    log(f"  Mean: {worst_true.mean():.4f}")
    log(f"  At 0: {(worst_true == 0).sum()}")
    log(f"  At 1: {(worst_true == 1).sum()}")
    log(f"  In (0,1): {((worst_true > 0) & (worst_true < 1)).sum()}")
    
    # =========================================================================
    # 5. TARGET DISTRIBUTION
    # =========================================================================
    log("\n[5. TARGET DISTRIBUTION]")
    target = df[target_col]
    log(f"Min: {target.min():.6f}")
    log(f"Max: {target.max():.6f}")
    log(f"Mean: {target.mean():.6f}")
    log(f"Median: {target.median():.6f}")
    log(f"Std: {target.std():.6f}")
    log(f"Skewness: {target.skew():.4f}")
    
    zero_pct = (target == 0).mean() * 100
    one_pct = (target == 1).mean() * 100
    log(f"\nExactly 0: {(target == 0).sum():,} ({zero_pct:.1f}%)")
    log(f"Exactly 1: {(target == 1).sum():,} ({one_pct:.1f}%)")
    log(f"In (0,1): {((target > 0) & (target < 1)).sum():,} ({((target > 0) & (target < 1)).mean()*100:.1f}%)")
    
    # =========================================================================
    # 6. FEATURE STATISTICS
    # =========================================================================
    log("\n[6. FEATURE STATISTICS (all features)]")
    log(f"{'Feature':<25} {'Min':>12} {'Max':>12} {'Mean':>12} {'Std':>12} {'Skew':>8} {'NaN%':>6}")
    
    for col in num_cols:
        if col in df.columns:
            s = df[col]
            nan_pct = s.isna().mean() * 100
            s_clean = s.dropna()
            if len(s_clean) > 0:
                log(f"{col:<25} {s_clean.min():>12.4f} {s_clean.max():>12.4f} {s_clean.mean():>12.4f} {s_clean.std():>12.4f} {s_clean.skew():>8.2f} {nan_pct:>5.1f}%")
    
    # =========================================================================
    # 7. FEATURE-TARGET CORRELATIONS
    # =========================================================================
    log("\n[7. FEATURE-TARGET CORRELATIONS]")
    correlations = {}
    for col in num_cols:
        if col in df.columns:
            corr = df[col].corr(target)
            if not np.isnan(corr):
                correlations[col] = corr
    
    sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    log(f"{'Rank':<5} {'Feature':<25} {'Correlation':>12}")
    for i, (feat, corr) in enumerate(sorted_corr, 1):
        log(f"{i:<5} {feat:<25} {corr:>+12.4f}")
    
    max_corr = max(abs(c) for _, c in sorted_corr) if sorted_corr else 0
    mean_corr = np.mean([abs(c) for _, c in sorted_corr]) if sorted_corr else 0
    log(f"\nMax |correlation|: {max_corr:.4f}")
    log(f"Mean |correlation|: {mean_corr:.4f}")
    
    # =========================================================================
    # 8. OUTLIER ANALYSIS
    # =========================================================================
    log("\n[8. OUTLIER ANALYSIS]")
    
    # Per-feature outliers (3 IQR)
    log("\nFeatures with outliers (>3 IQR from quartiles):")
    outlier_counts = {}
    for col in num_cols:
        if col in df.columns:
            s = df[col].dropna()
            if len(s) > 20:
                q1, q3 = s.quantile(0.25), s.quantile(0.75)
                iqr = q3 - q1
                if iqr > 0:
                    n_out = ((s < q1 - 3*iqr) | (s > q3 + 3*iqr)).sum()
                    if n_out > 0:
                        outlier_counts[col] = n_out
    
    for col, count in sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True):
        log(f"  {col:<25}: {count:>5} ({count/len(df)*100:.2f}%)")
    
    if not outlier_counts:
        log("  None detected")
    
    # Multi-feature outliers
    log("\nRows that are outliers (|z|>3) in multiple features:")
    row_outlier_counts = pd.Series(0, index=df.index)
    for col in num_cols:
        if col in df.columns:
            s = df[col]
            if s.std() > 0:
                z = (s - s.mean()) / s.std()
                row_outlier_counts += (z.abs() > 3).astype(int)
    
    multi_outliers = row_outlier_counts[row_outlier_counts >= 2].sort_values(ascending=False)
    if len(multi_outliers) > 0:
        log(f"Rows outlying in 2+ features: {len(multi_outliers)}")
        log(f"Max features outlying in single row: {row_outlier_counts.max()}")
        log(f"\n{'Row':>8} {'#OutlyingFeats':>15} {'Target':>10}")
        for idx, count in multi_outliers.head(20).items():
            target_val = df.loc[idx, target_col] if idx in df.index else np.nan
            log(f"{idx:>8} {count:>15} {target_val:>10.4f}")
    else:
        log("  No rows are outliers in multiple features")
    
    # =========================================================================
    # 9. PCA-SPECIFIC ANALYSIS
    # =========================================================================
    if is_pca_dataset:
        log("\n[9. PCA COMPONENT ANALYSIS]")
        pca_cols = [col for col in num_cols if col.startswith('pca_')]
        
        log(f"{'Component':<12} {'Min':>12} {'Max':>12} {'Range':>12} {'Mean':>10} {'Std':>10}")
        extreme_components = []
        for col in pca_cols:
            if col in df.columns:
                s = df[col]
                rng = s.max() - s.min()
                flag = " ⚠️" if rng > 100 else ""
                if rng > 100:
                    extreme_components.append((col, rng))
                log(f"{col:<12} {s.min():>12.2f} {s.max():>12.2f} {rng:>12.2f} {s.mean():>10.2f} {s.std():>10.2f}{flag}")
        
        if extreme_components:
            log(f"\n⚠️ {len(extreme_components)} components have range > 100")
            
            # Find extreme rows in PCA space
            log("\nExtreme values in PCA components:")
            for pca_col, rng in extreme_components[:5]:
                s = df[pca_col]
                log(f"\n{pca_col} (range={rng:.1f}):")
                
                # Highest values
                highest = s.nlargest(5)
                log(f"  Highest: {list(zip(highest.index.tolist(), highest.values.round(2).tolist()))}")
                
                # Lowest values
                lowest = s.nsmallest(5)
                log(f"  Lowest:  {list(zip(lowest.index.tolist(), lowest.values.round(2).tolist()))}")
                
                # Target values for these extreme rows
                extreme_rows = list(highest.index) + list(lowest.index)
                extreme_targets = df.loc[extreme_rows, target_col]
                log(f"  Target values for extreme rows: min={extreme_targets.min():.3f}, max={extreme_targets.max():.3f}, mean={extreme_targets.mean():.3f}")
    
    # =========================================================================
    # 10. SAMPLE DATA ROWS
    # =========================================================================
    log("\n[10. SAMPLE DATA ROWS]")
    
    # Show a few random rows
    log("\n5 Random rows (first 8 features + target):")
    sample_cols = num_cols[:8] + [target_col]
    sample_cols = [c for c in sample_cols if c in df.columns]
    sample_rows = df[sample_cols].sample(min(5, len(df)), random_state=42)
    log(sample_rows.to_string())
    
    # Show rows with extreme predictions
    if len(worst_indices) > 0:
        log("\n\nRows corresponding to worst predictions:")
        # We need to map test indices back to original df indices
        # This is approximate - showing similar target values
        worst_targets = all_y_true[worst_indices[:5]]
        log(f"Target values of worst-predicted test samples: {worst_targets.round(4).tolist()}")
    
    # =========================================================================
    # 11. FEATURE CORRELATIONS (INTER-FEATURE)
    # =========================================================================
    log("\n[11. INTER-FEATURE CORRELATIONS]")
    
    if len(num_cols) <= 20:
        # Show full correlation matrix for small datasets
        corr_matrix = df[num_cols].corr()
        
        # Find highly correlated pairs
        high_corr_pairs = []
        for i, col1 in enumerate(num_cols):
            for col2 in num_cols[i+1:]:
                if col1 in corr_matrix.columns and col2 in corr_matrix.columns:
                    c = corr_matrix.loc[col1, col2]
                    if abs(c) > 0.8:
                        high_corr_pairs.append((col1, col2, c))
        
        if high_corr_pairs:
            log("Highly correlated feature pairs (|r| > 0.8):")
            for col1, col2, c in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
                log(f"  {col1} <-> {col2}: {c:.4f}")
        else:
            log("No feature pairs with |r| > 0.8")
    else:
        log(f"Skipping full correlation matrix ({len(num_cols)} features)")
    
    # =========================================================================
    # 12. RESIDUAL ANALYSIS
    # =========================================================================
    log("\n[12. RESIDUAL ANALYSIS]")
    log(f"Raw residuals (true - raw_pred):")
    log(f"  Mean: {residuals_raw.mean():.6f}")
    log(f"  Std: {residuals_raw.std():.6f}")
    log(f"  Min: {residuals_raw.min():.4f}")
    log(f"  Max: {residuals_raw.max():.4f}")
    log(f"  Skewness: {pd.Series(residuals_raw).skew():.4f}")
    
    # Residuals by target value bins
    log("\nResiduals by target value:")
    bins = [0, 0.001, 0.1, 0.3, 0.5, 0.7, 0.9, 0.999, 1.0]
    for i in range(len(bins)-1):
        mask = (all_y_true >= bins[i]) & (all_y_true < bins[i+1])
        if mask.sum() > 0:
            bin_resid = residuals_raw[mask]
            log(f"  [{bins[i]:.3f}, {bins[i+1]:.3f}): n={mask.sum():>5}, mean_resid={bin_resid.mean():>+8.4f}, std={bin_resid.std():>8.4f}")
    
    # =========================================================================
    # 13. DIAGNOSIS
    # =========================================================================
    log("\n" + "="*80)
    log("DIAGNOSIS SUMMARY")
    log("="*80)
    
    issues = []
    r2 = agg['r2_mean']
    
    if r2 < 0:
        issues.append("🔴 CRITICAL: Negative R² - model worse than mean prediction")
    elif r2 < 0.05:
        issues.append(f"🔴 CRITICAL: Very low R² ({r2:.4f})")
    elif r2 < 0.15:
        issues.append(f"🟠 SEVERE: Low R² ({r2:.4f})")
    
    r2_std = np.std(agg['r2_scores'])
    if r2_std > 0.05:
        issues.append(f"🟠 SEVERE: High fold variance (R² std = {r2_std:.4f})")
    
    if max_corr < 0.2:
        issues.append(f"🔴 CRITICAL: Very weak linear signal (max |r| = {max_corr:.4f})")
    elif max_corr < 0.4:
        issues.append(f"🟠 SEVERE: Weak linear signal (max |r| = {max_corr:.4f})")
    
    extreme_pred_pct = ((all_y_pred_raw < -10).sum() + (all_y_pred_raw > 10).sum()) / len(all_y_pred_raw) * 100
    if extreme_pred_pct > 0.1:
        issues.append(f"🔴 CRITICAL: {extreme_pred_pct:.2f}% predictions outside [-10, 10]")
    
    clipping_pct = (total_clipped_below + total_clipped_above) / len(all_y_pred) * 100
    if clipping_pct > 20:
        issues.append(f"🟠 SEVERE: {clipping_pct:.1f}% predictions clipped")
    
    if zero_pct + one_pct > 40:
        issues.append(f"🟡 MODERATE: {zero_pct+one_pct:.0f}% target at boundaries (0 or 1)")
    
    if is_pca_dataset and 'extreme_components' in dir() and len(extreme_components) > 0:
        issues.append(f"🔴 CRITICAL: {len(extreme_components)} PCA components have extreme ranges (outlier amplification)")
    
    if len(multi_outliers) > len(df) * 0.01:
        issues.append(f"🟠 SEVERE: {len(multi_outliers)} rows ({len(multi_outliers)/len(df)*100:.1f}%) are multi-feature outliers")
    
    log("\nIssues found:")
    for issue in issues:
        log(f"  {issue}")
    
    if not issues:
        log("  No major issues detected")
    
    log("\n" + "="*80)
    
    return "\n".join(output)


# Run and print
for dataset in DATASETS_TO_DEBUG:
    if dataset in LGD_DATA:
        result = debug_lr_text_only(dataset)
        print(result)
    else:
        print(f"❌ '{dataset}' not found. Available: {list(LGD_DATA.keys())}")

In [ ]:
# =============================================================================
# COMPREHENSIVE BUG VERIFICATION: TALENT LinearRegression Denormalization
# =============================================================================
"""
This cell verifies that TALENT's LinearRegression returns predictions in 
NORMALIZED space rather than denormalizing back to original scale.

We will:
1. Calculate normalization parameters from training data
2. Manually denormalize predictions
3. Compare with target statistics
4. Test the denormalization formula
5. Verify this fixes all issues
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

print("="*80)
print("  TALENT LINEAR REGRESSION DENORMALIZATION BUG VERIFICATION")
print("="*80)

# =============================================================================
# SECTION 0: Load Data and Get Training Statistics
# =============================================================================
print("\n[0. LOADING DATA AND COMPUTING NORMALIZATION PARAMETERS]")
print("-" * 60)

# Load the dataset to get training statistics
from src.data.dataset_preprocessing import preprocess_dataset_specific

df, target_col, cat_cols, num_cols = preprocess_dataset_specific(
    task='lgd',
    dataset='0005.base_modelisation',
    apply_pca=False,
    remove_outliers=False
)

# These are the statistics TALENT would compute
y_train_full = df[target_col].values
mean_norm = y_train_full.mean()
std_norm = y_train_full.std()

print(f"Training set target statistics:")
print(f"  N samples: {len(y_train_full):,}")
print(f"  Mean:      {mean_norm:.6f}")
print(f"  Std:       {std_norm:.6f}")
print(f"  Range:     [{y_train_full.min():.6f}, {y_train_full.max():.6f}]")

print(f"\nTALENT normalization formula:")
print(f"  y_normalized = (y - {mean_norm:.6f}) / {std_norm:.6f}")
print(f"\nTo denormalize predictions:")
print(f"  y_original = y_normalized * {std_norm:.6f} + {mean_norm:.6f}")

# =============================================================================
# SECTION 1: Run LinearRegression and Get Predictions
# =============================================================================
print("\n[1. RUNNING LINEAR REGRESSION THROUGH TALENT]")
print("-" * 60)

from src.methods.method_runner import run_talent_method

results = run_talent_method(
    task='lgd',
    dataset='0001.heloc',
    test_size=0.2,
    val_size=0.1,
    cv_splits=5,
    seed=42,
    method='LinearRegression',
    tune=False,
    verbose=False
)

print(f"Completed {len(results)} folds")

# =============================================================================
# SECTION 2: Analyze Raw Predictions (Before Our Clipping)
# =============================================================================
print("\n[2. RAW PREDICTIONS FROM TALENT]")
print("-" * 60)

# Aggregate across all folds
all_y_true = np.concatenate([results[f]['y_true'] for f in results.keys()])
all_y_pred_raw = np.concatenate([results[f]['y_pred_raw'] for f in results.keys()])
all_y_pred_clipped = np.concatenate([results[f]['y_pred'] for f in results.keys()])

print(f"Target (ground truth) statistics:")
print(f"  Mean:   {all_y_true.mean():.6f}  ← Should match training mean {mean_norm:.6f}")
print(f"  Std:    {all_y_true.std():.6f}")
print(f"  Range:  [{all_y_true.min():.6f}, {all_y_true.max():.6f}]")

print(f"\nRaw predictions (from TALENT) statistics:")
print(f"  Mean:   {all_y_pred_raw.mean():.6f}  ← Should be ~{mean_norm:.3f}, but is ~0! 🔴")
print(f"  Std:    {all_y_pred_raw.std():.6f}  ← Should be ~{std_norm:.3f}")
print(f"  Range:  [{all_y_pred_raw.min():.6f}, {all_y_pred_raw.max():.6f}]")

print(f"\nClipped predictions (our post-processing):")
print(f"  Mean:   {all_y_pred_clipped.mean():.6f}")
print(f"  Std:    {all_y_pred_clipped.std():.6f}")
print(f"  Range:  [{all_y_pred_clipped.min():.6f}, {all_y_pred_clipped.max():.6f}]")

# Clipping statistics
n_clipped_below = (all_y_pred_raw < 0).sum()
n_clipped_above = (all_y_pred_raw > 1).sum()
pct_clipped = (n_clipped_below + n_clipped_above) / len(all_y_pred_raw) * 100

print(f"\nClipping impact:")
print(f"  {n_clipped_below:,} predictions < 0 ({n_clipped_below/len(all_y_pred_raw)*100:.1f}%)")
print(f"  {n_clipped_above:,} predictions > 1 ({n_clipped_above/len(all_y_pred_raw)*100:.1f}%)")
print(f"  Total clipped: {pct_clipped:.1f}%")

# =============================================================================
# SECTION 3: Test Denormalization Formula
# =============================================================================
print("\n[3. APPLYING DENORMALIZATION FORMULA]")
print("-" * 60)

# The denormalization formula: pred_original = pred_normalized * std + mean
y_pred_denormalized = all_y_pred_raw * std_norm + mean_norm

print(f"Formula: pred_denorm = pred_raw * {std_norm:.6f} + {mean_norm:.6f}")
print(f"\nDenormalized predictions:")
print(f"  Mean:   {y_pred_denormalized.mean():.6f}  ← Should match target mean {mean_norm:.6f} ✓")
print(f"  Std:    {y_pred_denormalized.std():.6f}")
print(f"  Range:  [{y_pred_denormalized.min():.6f}, {y_pred_denormalized.max():.6f}]")

# Check if mean matches
mean_diff = np.abs(y_pred_denormalized.mean() - all_y_true.mean())
mean_match = mean_diff < 0.05
print(f"\nMean difference: {mean_diff:.6f}")
print(f"Means match: {'✓ YES' if mean_match else '✗ NO'}")

# After denormalization, how much clipping is needed?
n_below_denorm = (y_pred_denormalized < 0).sum()
n_above_denorm = (y_pred_denormalized > 1).sum()
pct_clip_denorm = (n_below_denorm + n_above_denorm) / len(y_pred_denormalized) * 100

print(f"\nClipping needed after denormalization:")
print(f"  {n_below_denorm:,} predictions < 0 ({n_below_denorm/len(y_pred_denormalized)*100:.1f}%)")
print(f"  {n_above_denorm:,} predictions > 1 ({n_above_denorm/len(y_pred_denormalized)*100:.1f}%)")
print(f"  Total: {pct_clip_denorm:.1f}% (much better than {pct_clipped:.1f}% without denorm)")

# =============================================================================
# SECTION 4: Verify This Is Normalized Space
# =============================================================================
print("\n[4. STATISTICAL TESTS FOR NORMALIZATION]")
print("-" * 60)

# Test 1: Is raw prediction mean close to 0?
test1_pass = np.abs(all_y_pred_raw.mean()) < 0.05
print(f"Test 1: Raw predictions centered at 0 (normalized space indicator)")
print(f"  Mean = {all_y_pred_raw.mean():.6f}")
print(f"  Expected: ~0.00 (normalized space)")
print(f"  Result: {'✓ PASS' if test1_pass else '✗ FAIL'} (mean close to 0)")

# Test 2: Is raw prediction std close to target std (or close to 1)?
# In normalized space, std should be ~1, but after LR it might differ
test2_pass = 0.3 < all_y_pred_raw.std() < 1.5  # Reasonable range for normalized space
print(f"\nTest 2: Raw predictions have std consistent with normalized space")
print(f"  Std = {all_y_pred_raw.std():.6f}")
print(f"  Expected: ~1.0 (normalized space) or ~{std_norm:.3f} (original space)")
print(f"  Result: {'✓ PASS' if test2_pass else '✗ FAIL'} (std in normalized range)")

# Test 3: After denormalization, does mean match target?
test3_pass = np.abs(y_pred_denormalized.mean() - all_y_true.mean()) < 0.1
print(f"\nTest 3: Denormalized predictions match target mean")
print(f"  Denorm mean = {y_pred_denormalized.mean():.6f}")
print(f"  Target mean = {all_y_true.mean():.6f}")
print(f"  Difference  = {np.abs(y_pred_denormalized.mean() - all_y_true.mean()):.6f}")
print(f"  Result: {'✓ PASS' if test3_pass else '✗ FAIL'} (means match within 0.1)")

# Test 4: Are raw predictions in wrong range for [0,1] target?
pct_outside = ((all_y_pred_raw < 0) | (all_y_pred_raw > 1)).mean() * 100
test4_pass = pct_outside > 10  # More than 10% outside [0,1]
print(f"\nTest 4: Raw predictions outside valid [0,1] range")
print(f"  % outside [0,1]: {pct_outside:.1f}%")
print(f"  Expected: >10% (indicates wrong scale)")
print(f"  Result: {'✓ PASS' if test4_pass else '✗ FAIL'} (many predictions invalid)")

# Test 5: Linear relationship between normalized predictions and target
# If predictions are normalized but target is not, correlation should be low
corr_raw = np.corrcoef(all_y_pred_raw, all_y_true)[0, 1]
corr_denorm = np.corrcoef(y_pred_denormalized, all_y_true)[0, 1]
test5_pass = corr_denorm > corr_raw  # Denorm should have same/better correlation
print(f"\nTest 5: Correlation analysis")
print(f"  Corr(raw, target):    {corr_raw:.4f}")
print(f"  Corr(denorm, target): {corr_denorm:.4f}")
print(f"  Result: {'✓ PASS' if test5_pass else '✗ FAIL'} (denorm correlation >= raw)")

# =============================================================================
# SECTION 5: Compare Metrics Before/After Denormalization
# =============================================================================
print("\n[5. METRICS COMPARISON]")
print("-" * 60)

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Scenario 1: Raw predictions vs Ground truth (WRONG - different scales)
print("Scenario 1: Raw Predictions vs Ground Truth")
print("  (This compares normalized predictions to original-scale targets - WRONG)")
r2_raw = r2_score(all_y_true, all_y_pred_raw)
mse_raw = mean_squared_error(all_y_true, all_y_pred_raw)
mae_raw = mean_absolute_error(all_y_true, all_y_pred_raw)
print(f"  R²:  {r2_raw:.6f}")
print(f"  MSE: {mse_raw:.6f}")
print(f"  MAE: {mae_raw:.6f}")

# Scenario 2: Clipped predictions (our current approach - STILL WRONG)
print("\nScenario 2: Clipped Predictions vs Ground Truth")
print("  (This clips normalized predictions to [0,1] - STILL WRONG)")
r2_clip = r2_score(all_y_true, all_y_pred_clipped)
mse_clip = mean_squared_error(all_y_true, all_y_pred_clipped)
mae_clip = mean_absolute_error(all_y_true, all_y_pred_clipped)
print(f"  R²:  {r2_clip:.6f}  ← Negative! Model worse than mean baseline")
print(f"  MSE: {mse_clip:.6f}")
print(f"  MAE: {mae_clip:.6f}")

# Scenario 3: Denormalized predictions (CORRECT approach)
y_pred_denorm_clipped = np.clip(y_pred_denormalized, 0, 1)
print("\nScenario 3: Denormalized then Clipped vs Ground Truth")
print("  (This denormalizes FIRST, then clips - CORRECT)")
r2_denorm = r2_score(all_y_true, y_pred_denorm_clipped)
mse_denorm = mean_squared_error(all_y_true, y_pred_denorm_clipped)
mae_denorm = mean_absolute_error(all_y_true, y_pred_denorm_clipped)
print(f"  R²:  {r2_denorm:.6f}  ← Much better! Positive R²")
print(f"  MSE: {mse_denorm:.6f}")
print(f"  MAE: {mae_denorm:.6f}")

improvement = r2_denorm - r2_clip
print(f"\n🎯 R² improvement with denormalization:")
print(f"  Absolute: {improvement:.6f}")
if r2_clip < 0:
    print(f"  Goes from NEGATIVE to {'POSITIVE' if r2_denorm > 0 else 'LESS NEGATIVE'}")
else:
    print(f"  Relative: {improvement/r2_clip*100:.1f}%")

# Compare to predicting mean (baseline)
y_mean_baseline = np.full_like(all_y_true, all_y_true.mean())
r2_baseline = r2_score(all_y_true, y_mean_baseline)
print(f"\nBaseline (predicting mean): R² = {r2_baseline:.6f}")
print(f"  Scenario 2 (clipped): {'WORSE' if r2_clip < r2_baseline else 'BETTER'} than baseline")
print(f"  Scenario 3 (denorm):  {'WORSE' if r2_denorm < r2_baseline else 'BETTER'} than baseline")

# =============================================================================
# SECTION 6: Detailed Statistical Analysis
# =============================================================================
print("\n[6. DETAILED STATISTICAL ANALYSIS]")
print("-" * 60)

# Distribution comparison
print("Distribution comparison:")
print(f"  Target:           mean={all_y_true.mean():.4f}, std={all_y_true.std():.4f}")
print(f"  Raw predictions:  mean={all_y_pred_raw.mean():.4f}, std={all_y_pred_raw.std():.4f}")
print(f"  Denorm pred:      mean={y_pred_denormalized.mean():.4f}, std={y_pred_denormalized.std():.4f}")

# Kolmogorov-Smirnov test: Are distributions similar?
from scipy.stats import ks_2samp
ks_raw, p_raw = ks_2samp(all_y_true, all_y_pred_raw)
ks_denorm, p_denorm = ks_2samp(all_y_true, y_pred_denormalized)

print(f"\nKolmogorov-Smirnov test (distributions similar if KS small, p large):")
print(f"  Raw vs Target:    KS={ks_raw:.4f}, p={p_raw:.6f}")
print(f"  Denorm vs Target: KS={ks_denorm:.4f}, p={p_denorm:.6f}")
print(f"  {'✓' if ks_denorm < ks_raw else '✗'} Denorm has {'better' if ks_denorm < ks_raw else 'worse'} distribution match")

# Quantile analysis
quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]
print(f"\nQuantile analysis:")
print(f"  {'Quantile':<10} {'Target':<10} {'Raw':<10} {'Denorm':<10}")
for q in quantiles:
    q_true = np.quantile(all_y_true, q)
    q_raw = np.quantile(all_y_pred_raw, q)
    q_denorm = np.quantile(y_pred_denormalized, q)
    print(f"  {q:<10.2f} {q_true:<10.4f} {q_raw:<10.4f} {q_denorm:<10.4f}")

# =============================================================================
# SECTION 7: Visualizations
# =============================================================================
print("\n[7. GENERATING DIAGNOSTIC PLOTS]")
print("-" * 60)

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
fig.suptitle('TALENT LinearRegression Denormalization Bug Verification', 
             fontsize=16, fontweight='bold')

# Row 1: Scatter plots
# Plot 1: Raw predictions (normalized space)
ax = axes[0, 0]
ax.scatter(all_y_true, all_y_pred_raw, alpha=0.3, s=5, c='red')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect prediction', linewidth=2)
ax.axhline(y=0, color='orange', linestyle=':', alpha=0.7, label='y=0')
ax.axhline(y=1, color='orange', linestyle=':', alpha=0.7, label='y=1')
ax.set_xlabel('True LGD', fontsize=11)
ax.set_ylabel('Raw Prediction (normalized)', fontsize=11)
ax.set_title(f'❌ Raw Predictions\nR²={r2_raw:.4f}', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-2.5, 1.5)

# Plot 2: Denormalized predictions
ax = axes[0, 1]
ax.scatter(all_y_true, y_pred_denormalized, alpha=0.3, s=5, color='blue')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect prediction', linewidth=2)
ax.axhline(y=0, color='orange', linestyle=':', alpha=0.7)
ax.axhline(y=1, color='orange', linestyle=':', alpha=0.7)
ax.set_xlabel('True LGD', fontsize=11)
ax.set_ylabel('Denormalized Prediction', fontsize=11)
ax.set_title('⚠️ Denormalized (Before Clipping)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(-0.1, 1.1)

# Plot 3: Denormalized + clipped (CORRECT)
ax = axes[0, 2]
ax.scatter(all_y_true, y_pred_denorm_clipped, alpha=0.3, s=5, color='green')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect prediction', linewidth=2)
ax.set_xlabel('True LGD', fontsize=11)
ax.set_ylabel('Denormalized + Clipped', fontsize=11)
ax.set_title(f'✅ CORRECT: Denorm + Clip\nR²={r2_denorm:.4f}', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)

# Row 2: Distributions
# Plot 4: Raw predictions distribution
ax = axes[1, 0]
ax.hist(all_y_true, bins=50, alpha=0.5, label=f'True (μ={all_y_true.mean():.3f})', 
        color='blue', edgecolor='white', density=True)
ax.hist(all_y_pred_raw, bins=50, alpha=0.5, label=f'Raw (μ={all_y_pred_raw.mean():.3f})', 
        color='red', edgecolor='white', density=True)
ax.axvline(all_y_true.mean(), color='blue', linestyle='--', linewidth=2, alpha=0.8)
ax.axvline(all_y_pred_raw.mean(), color='red', linestyle='--', linewidth=2, alpha=0.8)
ax.set_xlabel('Value', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('❌ Distribution: True vs Raw', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Plot 5: Denormalized distribution
ax = axes[1, 1]
ax.hist(all_y_true, bins=50, alpha=0.5, label=f'True (μ={all_y_true.mean():.3f})', 
        color='blue', edgecolor='white', density=True)
ax.hist(y_pred_denormalized, bins=50, alpha=0.5, 
        label=f'Denorm (μ={y_pred_denormalized.mean():.3f})', 
        color='green', edgecolor='white', density=True)
ax.axvline(all_y_true.mean(), color='blue', linestyle='--', linewidth=2, alpha=0.8)
ax.axvline(y_pred_denormalized.mean(), color='green', linestyle='--', linewidth=2, alpha=0.8)
ax.set_xlabel('Value', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('✅ Distribution: True vs Denormalized', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Plot 6: Q-Q plot
ax = axes[1, 2]
from scipy import stats as sp_stats
sp_stats.probplot(all_y_pred_raw, dist="norm", plot=ax)
ax.set_title('Q-Q Plot: Raw Predictions\n(Should be normal if in normalized space)', 
             fontsize=11, fontweight='bold')
ax.grid(alpha=0.3)

# Row 3: Residuals and errors
# Plot 7: Residuals comparison
ax = axes[2, 0]
residuals_wrong = all_y_true - all_y_pred_clipped
residuals_correct = all_y_true - y_pred_denorm_clipped
ax.hist(residuals_wrong, bins=50, alpha=0.5, 
        label=f'Without Denorm (std={residuals_wrong.std():.3f})', 
        color='red', edgecolor='white')
ax.hist(residuals_correct, bins=50, alpha=0.5, 
        label=f'With Denorm (std={residuals_correct.std():.3f})', 
        color='green', edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(residuals_wrong.mean(), color='red', linestyle=':', linewidth=2, alpha=0.7)
ax.axvline(residuals_correct.mean(), color='green', linestyle=':', linewidth=2, alpha=0.7)
ax.set_xlabel('Residual', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Residual Distribution Comparison', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 8: Absolute errors by true value
ax = axes[2, 1]
abs_err_wrong = np.abs(residuals_wrong)
abs_err_correct = np.abs(residuals_correct)
ax.scatter(all_y_true, abs_err_wrong, alpha=0.3, s=5, color='red', label='Without denorm')
ax.scatter(all_y_true, abs_err_correct, alpha=0.3, s=5, color='green', label='With denorm')
ax.set_xlabel('True LGD', fontsize=11)
ax.set_ylabel('Absolute Error', fontsize=11)
ax.set_title('Absolute Error by True Value', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Plot 9: Metrics comparison bar chart
ax = axes[2, 2]
metrics_names = ['R²', 'MSE', 'MAE']
metrics_wrong = [r2_clip, mse_clip, mae_clip]
metrics_correct = [r2_denorm, mse_denorm, mae_denorm]

x = np.arange(len(metrics_names))
width = 0.35

bars1 = ax.bar(x - width/2, metrics_wrong, width, label='Without Denorm', 
               color='red', alpha=0.7)
bars2 = ax.bar(x + width/2, metrics_correct, width, label='With Denorm', 
               color='green', alpha=0.7)

ax.set_ylabel('Metric Value', fontsize=11)
ax.set_title('Metrics Comparison', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.grid(alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

# =============================================================================
# SECTION 8: Summary Statistics Table
# =============================================================================
print("\n[8. SUMMARY STATISTICS TABLE]")
print("-" * 60)

summary_df = pd.DataFrame({
    'Statistic': ['Mean', 'Std', 'Min', 'Max', '10th pct', '90th pct', 'R²', 'MSE', 'MAE'],
    'Ground Truth': [
        all_y_true.mean(),
        all_y_true.std(),
        all_y_true.min(),
        all_y_true.max(),
        np.percentile(all_y_true, 10),
        np.percentile(all_y_true, 90),
        np.nan,
        np.nan,
        np.nan
    ],
    'Raw (Normalized)': [
        all_y_pred_raw.mean(),
        all_y_pred_raw.std(),
        all_y_pred_raw.min(),
        all_y_pred_raw.max(),
        np.percentile(all_y_pred_raw, 10),
        np.percentile(all_y_pred_raw, 90),
        r2_raw,
        mse_raw,
        mae_raw
    ],
    'Clipped (Wrong)': [
        all_y_pred_clipped.mean(),
        all_y_pred_clipped.std(),
        all_y_pred_clipped.min(),
        all_y_pred_clipped.max(),
        np.percentile(all_y_pred_clipped, 10),
        np.percentile(all_y_pred_clipped, 90),
        r2_clip,
        mse_clip,
        mae_clip
    ],
    'Denorm + Clip (Correct)': [
        y_pred_denorm_clipped.mean(),
        y_pred_denorm_clipped.std(),
        y_pred_denorm_clipped.min(),
        y_pred_denorm_clipped.max(),
        np.percentile(y_pred_denorm_clipped, 10),
        np.percentile(y_pred_denorm_clipped, 90),
        r2_denorm,
        mse_denorm,
        mae_denorm
    ]
})

print(summary_df.to_string(index=False))

# =============================================================================
# SECTION 9: Final Verdict
# =============================================================================
print("\n" + "="*80)
print("  FINAL VERDICT")
print("="*80)

all_tests_pass = test1_pass and test2_pass and test3_pass and test4_pass and test5_pass

if all_tests_pass:
    print("\n🔴 BUG CONFIRMED 🔴")
    print("\n✓ All 5 tests passed:")
    print("  1. ✓ Raw predictions centered at 0 (normalized space)")
    print("  2. ✓ Raw predictions have normalized-range variance")
    print("  3. ✓ Denormalized predictions match target mean")
    print("  4. ✓ Many raw predictions outside valid [0,1] range")
    print("  5. ✓ Denormalization improves correlation")
    
    print(f"\n📊 Quantitative Evidence:")
    print(f"  • R² improves from {r2_clip:.4f} to {r2_denorm:.4f}")
    print(f"  • Prediction mean shifts from {all_y_pred_raw.mean():.4f} to {y_pred_denormalized.mean():.4f}")
    print(f"  • Target mean is {all_y_true.mean():.4f}")
    print(f"  • Clipping reduced from {pct_clipped:.1f}% to {pct_clip_denorm:.1f}%")
    
    print("\n🎯 Conclusion:")
    print("  TALENT's LinearRegression returns predictions in NORMALIZED space.")
    print("  Formula: y_normalized = (y - mean) / std")
    print("  Predictions must be denormalized: y_original = y_normalized * std + mean")
    print("  This is a confirmed bug in TALENT affecting all regression tasks.")
else:
    print("\n✗ Not all tests passed - review individual test results above")
    print(f"  Test 1 (mean ~0): {'PASS' if test1_pass else 'FAIL'}")
    print(f"  Test 2 (std normalized): {'PASS' if test2_pass else 'FAIL'}")
    print(f"  Test 3 (denorm matches): {'PASS' if test3_pass else 'FAIL'}")
    print(f"  Test 4 (many outside [0,1]): {'PASS' if test4_pass else 'FAIL'}")
    print(f"  Test 5 (correlation improves): {'PASS' if test5_pass else 'FAIL'}")

print("\n" + "="*80)
print("  RECOMMENDED ACTIONS")
print("="*80)
print("""
1. For TALENT developers (GitHub issue):
   - Modify LinearRegressionMethod.predict() to denormalize before returning
   - Add: if y_info['policy'] == 'mean_std': 
            predictions = predictions * y_info['std'] + y_info['mean']
   - Audit ALL regression methods (RandomForest, XGBoost, etc.) for same issue

2. For our research (workaround in method_runner.py):
   - Store y_info (mean, std) in results dict
   - Add denormalization function
   - Apply before clipping: y_denorm = y_raw * std + mean
   - Then clip: y_final = np.clip(y_denorm, 0, 1)

3. For paper/presentation:
   - Document this bug discovery in methodology section
   - Explain why results differ from naive TALENT usage
   - Note that other researchers using TALENT for regression may have this issue
""")

print("\n" + "="*80)

# Save results for later reference
results_dict = {
    'mean_norm': mean_norm,
    'std_norm': std_norm,
    'r2_without_denorm': r2_clip,
    'r2_with_denorm': r2_denorm,
    'improvement': r2_denorm - r2_clip,
    'pct_clipped_before': pct_clipped,
    'pct_clipped_after': pct_clip_denorm,
    'all_tests_pass': all_tests_pass
}

print(f"\n💾 Results saved to 'results_dict' variable for reference")

In [ ]:
# =============================================================================
# COMPREHENSIVE BUG VERIFICATION: ALL TALENT CLASSICAL REGRESSION METHODS
# =============================================================================
"""
This cell tests ALL classical regression methods in TALENT to verify the
denormalization bug. Method names must match TALENT's exact naming convention.
"""
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from src.methods.method_runner import run_talent_method
from src.data.dataset_preprocessing import preprocess_dataset_specific

# =============================================================================
# CONFIGURATION: Easily adjust these parameters
# =============================================================================
TASK = 'lgd'  # 'pd' or 'lgd'
DATASET = '0005.base_modelisation'  # e.g., '0001.heloc'
CV_SPLITS = 1
TEST_SIZE = 0.2
VAL_SIZE = 0.1
SEED = 42
APPLY_PCA = False
REMOVE_OUTLIERS = False

# Methods to test (method_name, tune)
METHODS_TO_TEST = [
    ('LinearRegression', False),
    ('knn', False),
    ('RandomForest', False),
    ('xgboost', False),
    ('lightgbm', False),
    ('catboost', False),
    ('svm', False),
]

print("="*80)
print("  TESTING ALL CLASSICAL REGRESSION METHODS FOR DENORMALIZATION BUG")
print("="*80)
print(f"\n[CONFIGURATION]")
print(f"  Task: {TASK.upper()}")
print(f"  Dataset: {DATASET}")
print(f"  CV Splits: {CV_SPLITS}")
print(f"  Test/Val Split: {TEST_SIZE}/{VAL_SIZE}")
print(f"  Random Seed: {SEED}")

# =============================================================================
# SETUP: Get normalization parameters from dataset
# =============================================================================
print("\n[SETUP: Loading dataset and computing normalization parameters]")
print("-" * 60)

df, target_col, cat_cols, num_cols = preprocess_dataset_specific(
    task=TASK,
    dataset=DATASET,
    apply_pca=APPLY_PCA,
    remove_outliers=REMOVE_OUTLIERS
)

y_train_full = df[target_col].values
mean_norm = y_train_full.mean()
std_norm = y_train_full.std()

print(f"Dataset: {DATASET} ({TASK.upper()} task)")
print(f"Target statistics:")
print(f"  N samples: {len(y_train_full):,}")
print(f"  Mean: {mean_norm:.6f}")
print(f"  Std:  {std_norm:.6f}")
print(f"  Range: [{y_train_full.min():.6f}, {y_train_full.max():.6f}]")
print(f"\nTALENT normalization: y_norm = (y - {mean_norm:.6f}) / {std_norm:.6f}")
print(f"Denormalization formula: y_orig = y_norm * {std_norm:.6f} + {mean_norm:.6f}")

# =============================================================================
# DEFINE METHODS TO TEST
# =============================================================================
print(f"\n[TESTING {len(METHODS_TO_TEST)} METHODS]")
print("-" * 60)
for method, tune in METHODS_TO_TEST:
    print(f"  • {method} (tune={tune})")

# =============================================================================
# TEST EACH METHOD
# =============================================================================

results_summary = []

for method_name, tune in METHODS_TO_TEST:
    print("\n" + "="*80)
    print(f"  TESTING: {method_name}")
    print("="*80)
    
    try:
        # Run the method
        print(f"\n[1. Running {method_name}...]")
        results = run_talent_method(
            task=TASK,
            dataset=DATASET,
            test_size=TEST_SIZE,
            val_size=VAL_SIZE,
            cv_splits=CV_SPLITS,
            seed=SEED,
            method=method_name,
            tune=tune,
            verbose=False
        )
        
        print(f"   ✓ Completed {len(results)} folds")
        
        # Aggregate predictions across folds
        all_y_true = np.concatenate([results[f]['y_true'] for f in results.keys()])
        all_y_pred_raw = np.concatenate([results[f]['y_pred_raw'] for f in results.keys()])
        all_y_pred_clipped = np.concatenate([results[f]['y_pred'] for f in results.keys()])
        
        # =================================================================
        # ANALYSIS: Check for denormalization bug
        # =================================================================
        print(f"\n[2. Analyzing predictions...]")
        
        # Test 1: Are raw predictions centered at 0? (normalized space indicator)
        raw_mean = all_y_pred_raw.mean()
        test1_pass = np.abs(raw_mean) < 0.1
        print(f"\n   Test 1: Raw predictions centered at 0")
        print(f"     Raw mean: {raw_mean:.6f} (expected: ~0.00)")
        print(f"     Result: {'✓ PASS' if test1_pass else '✗ FAIL'} - {'In normalized space' if test1_pass else 'Already denormalized?'}")
        
        # Test 2: Are predictions in wrong range for [0,1] target?
        pct_outside = ((all_y_pred_raw < 0) | (all_y_pred_raw > 1)).mean() * 100
        test2_pass = pct_outside > 10
        print(f"\n   Test 2: Predictions outside valid [0,1] range")
        print(f"     % outside [0,1]: {pct_outside:.1f}%")
        print(f"     Result: {'✓ PASS' if test2_pass else '✗ FAIL'} - {'Bug confirmed' if test2_pass else 'Possibly fixed'}")
        
        # Test 3: Does denormalization fix the mean?
        y_pred_denormalized = all_y_pred_raw * std_norm + mean_norm
        denorm_mean = y_pred_denormalized.mean()
        mean_match = np.abs(denorm_mean - all_y_true.mean()) < 0.1
        print(f"\n   Test 3: Denormalization fixes mean")
        print(f"     True mean: {all_y_true.mean():.6f}")
        print(f"     Denorm mean: {denorm_mean:.6f}")
        print(f"     Result: {'✓ PASS' if mean_match else '✗ FAIL'} - {'Bug confirmed' if mean_match else 'Unexpected'}")
        
        # Compute metrics
        r2_raw = r2_score(all_y_true, all_y_pred_raw)
        r2_clipped = r2_score(all_y_true, all_y_pred_clipped)
        
        y_pred_denorm_clipped = np.clip(y_pred_denormalized, 0, 1)
        r2_denorm = r2_score(all_y_true, y_pred_denorm_clipped)
        
        improvement = r2_denorm - r2_clipped
        
        print(f"\n[3. Performance Metrics]")
        print(f"     R² (raw vs true):     {r2_raw:.6f}")
        print(f"     R² (clipped vs true): {r2_clipped:.6f}")
        print(f"     R² (denorm vs true):  {r2_denorm:.6f}")
        print(f"     Improvement:          {improvement:.6f}")
        if r2_clipped != 0:
            print(f"     Relative improvement: {improvement/abs(r2_clipped)*100:.1f}%")
        
        # =================================================================
        # VERDICT
        # =================================================================
        bug_confirmed = test1_pass and test2_pass and mean_match
        
        print(f"\n[4. VERDICT FOR {method_name}]")
        if bug_confirmed:
            print(f"     🔴 BUG CONFIRMED 🔴")
            print(f"     This method returns predictions in NORMALIZED space.")
            print(f"     Predictions must be manually denormalized.")
        else:
            print(f"     ⚠️  UNEXPECTED RESULTS")
            print(f"     This method may already be denormalized or has different behavior.")
            print(f"     Manual inspection recommended.")
        
        # Store results
        results_summary.append({
            'Method': method_name,
            'Test1_CenteredAt0': '✓' if test1_pass else '✗',
            'Test2_OutsideRange': f'{pct_outside:.1f}%',
            'Test3_DenormFixesMean': '✓' if mean_match else '✗',
            'R²_without_denorm': r2_clipped,
            'R²_with_denorm': r2_denorm,
            'Improvement': improvement,
            'Bug_Confirmed': '🔴 YES' if bug_confirmed else '⚠️  CHECK',
        })
        
    except Exception as e:
        print(f"\n   ✗ ERROR running {method_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        results_summary.append({
            'Method': method_name,
            'Test1_CenteredAt0': 'ERROR',
            'Test2_OutsideRange': 'ERROR',
            'Test3_DenormFixesMean': 'ERROR',
            'R²_without_denorm': np.nan,
            'R²_with_denorm': np.nan,
            'Improvement': np.nan,
            'Bug_Confirmed': '❌ ERROR',
        })

# =============================================================================
# SUMMARY TABLE
# =============================================================================
print("\n" + "="*80)
print("  SUMMARY: DENORMALIZATION BUG ACROSS ALL METHODS")
print("="*80)
print(f"  Dataset: {DATASET} ({TASK.upper()})")
print("="*80)

summary_df = pd.DataFrame(results_summary)
print("\n" + summary_df.to_string(index=False))

# =============================================================================
# VISUALIZATION: R² COMPARISON
# =============================================================================
print("\n[Generating visualization...]")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Denormalization Bug Analysis: {DATASET} ({TASK.upper()})', 
             fontsize=16, fontweight='bold', y=1.02)

# Plot 1: R² Comparison
ax = axes[0]
methods = [r['Method'] for r in results_summary if not pd.isna(r['R²_without_denorm'])]
r2_without = [r['R²_without_denorm'] for r in results_summary if not pd.isna(r['R²_without_denorm'])]
r2_with = [r['R²_with_denorm'] for r in results_summary if not pd.isna(r['R²_with_denorm'])]

x = np.arange(len(methods))
width = 0.35

bars1 = ax.bar(x - width/2, r2_without, width, label='Without Denorm', 
               color='red', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, r2_with, width, label='With Denorm', 
               color='green', alpha=0.7, edgecolor='black')

ax.set_xlabel('Method', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('R² Score: Without vs With Denormalization', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom' if height > 0 else 'top', 
                fontsize=9, fontweight='bold')

# Plot 2: Improvement
ax = axes[1]
improvements = [r['Improvement'] for r in results_summary if not pd.isna(r['Improvement'])]
colors = ['green' if imp > 0 else 'red' for imp in improvements]

bars = ax.bar(methods, improvements, color=colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Method', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Improvement', fontsize=12, fontweight='bold')
ax.set_title('R² Improvement with Denormalization', fontsize=14, fontweight='bold')
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.grid(alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)

# Add value labels
for bar, imp in zip(bars, improvements):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{imp:.3f}',
            ha='center', va='bottom' if height > 0 else 'top',
            fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# =============================================================================
# FINAL RECOMMENDATIONS
# =============================================================================
print("\n" + "="*80)
print("  RECOMMENDATIONS")
print("="*80)

bug_count = sum(1 for r in results_summary if r['Bug_Confirmed'] == '🔴 YES')
total_count = len([r for r in results_summary if r['Bug_Confirmed'] != '❌ ERROR'])

print(f"\n📊 Summary:")
print(f"   Dataset: {DATASET} ({TASK.upper()})")
print(f"   {bug_count}/{total_count} methods confirmed with denormalization bug")

if bug_count > 0:
    print(f"\n🔴 CRITICAL: Multiple methods affected!")
    print(f"\n   Affected methods:")
    for r in results_summary:
        if r['Bug_Confirmed'] == '🔴 YES':
            print(f"      • {r['Method']}: R² improves by {r['Improvement']:.3f}")
    
    print(f"\n💡 Recommended Actions:")
    print(f"   1. Submit GitHub issue to TALENT repository")
    print(f"   2. Email TALENT authors with bug report")
    print(f"   3. Implement workaround in method_runner.py")
    print(f"   4. Re-run all LGD experiments with fix")
    print(f"   5. Document in research methodology")

print("\n" + "="*80)
print("  ANALYSIS COMPLETE")
print("="*80)